In [ ]:
# ── 1. INSTALAR DEPENDÊNCIAS ─────────────────────────────────
!pip install ultralytics supervision -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 12.2 MB/s eta 0:00:00


In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="doguilmak/Drone-Detection-YOLOv11x",
    filename="weight/best.pt"
)

print(model_path)

/root/.cache/huggingface/hub/models--doguilmak--Drone-Detection-YOLOv11x/snapshots/07422013c8c07ac6c0f0690a7b2cf8954d4ecc11/weight/best.pt


In [ ]:
# ============================================================
#  DRONE / UAV TRACKER — Google Colab
#  YOLOv11 + Military HUD (estilo míssil/seeker) + ByteTrack
#  Mira SEGUE o drone com suavização — sem filtro de cor
# ============================================================

import cv2
import numpy as np
from ultralytics import YOLO
import time
from datetime import datetime, timezone
from collections import defaultdict

# ── CONFIGURAÇÕES ───────────────────────────────────────────
MODEL        = "/root/.cache/huggingface/hub/models--doguilmak--Drone-Detection-YOLOv11x/snapshots/07422013c8c07ac6c0f0690a7b2cf8954d4ecc11/weight/best.pt"
CONF         = 0.35
SOURCE       = "/content/mixkit-drone-flying-with-the-sky-in-the-background-44642-hd-ready.mp4"
OUTPUT_VIDEO = "/content/uav_tracking_result.mp4"

TRAIL_LEN    = 50
SEEKER_W     = 320
SEEKER_H     = 240
ZOOM_PAD     = 0.8

# Suavização da mira:  0.0 = mira colada no drone (instantâneo)
#                      0.85 = mira muito lenta / inercial
# ~0.75 dá efeito servo / câmera de míssil real
AIM_SMOOTH   = 0.75

# ── PALETA ───────────────────────────────────────────────────
G_BRIGHT = (0, 255, 0)
G_MID    = (0, 200, 0)
G_DIM    = (0, 120, 0)
BLACK    = (0,   0, 0)

FONT    = cv2.FONT_HERSHEY_PLAIN
FONT_SM = 0.85
FONT_MD = 0.95

# ── ESTADO GLOBAL ────────────────────────────────────────────
trails   = defaultdict(list)
_aim_x   = -1.0   # posição suavizada da mira (float para precisão)
_aim_y   = -1.0

print("Carregando modelo YOLO...")
model = YOLO(MODEL)
print("✓ Modelo carregado")


# ── UTILITÁRIOS ──────────────────────────────────────────────
def put_text(img, text, pos, color=None, scale=FONT_SM, thickness=1):
    if color is None: color = G_BRIGHT
    cv2.putText(img, text, pos, FONT, scale, color, thickness, cv2.LINE_AA)

def corner_rect(img, x1, y1, x2, y2, color=None, arm=18, thick=2):
    if color is None: color = G_BRIGHT
    for (px,py),(dx,dy) in [((x1,y1),(1,1)),((x2,y1),(-1,1)),
                             ((x1,y2),(1,-1)),((x2,y2),(-1,-1))]:
        cv2.line(img,(px,py),(px+dx*arm,py),color,thick,cv2.LINE_AA)
        cv2.line(img,(px,py),(px,py+dy*arm),color,thick,cv2.LINE_AA)

def dashed_rect(img, x1, y1, x2, y2, color=None, gap=8, thick=1):
    if color is None: color = G_DIM
    for (ax,ay),(bx,by) in [((x1,y1),(x2,y1)),((x2,y1),(x2,y2)),
                              ((x2,y2),(x1,y2)),((x1,y2),(x1,y1))]:
        length = max(2,(abs(bx-ax)+abs(by-ay))//gap)
        pts = np.linspace([ax,ay],[bx,by],length).astype(int)
        for j in range(0,len(pts)-1,2):
            cv2.line(img,tuple(pts[j]),tuple(pts[j+1]),color,thick,cv2.LINE_AA)

def draw_trail(img, points):
    n = len(points)
    for i in range(1,n):
        a = i/n
        col = (0,int(255*a),0)
        cv2.line(img,points[i-1],points[i],col,max(1,int(a*2)),cv2.LINE_AA)


# ── CROSSHAIR — POSIÇÃO DINÂMICA ─────────────────────────────
def draw_crosshair(img, cx, cy, w, h, locked=False):
    """
    Crosshair que se move para (cx, cy).
    locked=True: linhas chegam até as bordas da tela (modo engajamento).
    locked=False: linhas curtas / discretas (modo busca).
    """
    # linhas longas tênues do centro para as bordas
    cv2.line(img, (0, cy), (w, cy), G_DIM, 1, cv2.LINE_AA)
    cv2.line(img, (cx, 0), (cx, h), G_DIM, 1, cv2.LINE_AA)

    gap  = 28
    arm  = 80 if locked else 55
    col  = G_BRIGHT

    # segmentos com gap central
    cv2.line(img, (cx-arm-gap, cy), (cx-gap, cy), col, 2, cv2.LINE_AA)
    cv2.line(img, (cx+gap, cy),     (cx+arm+gap, cy), col, 2, cv2.LINE_AA)
    cv2.line(img, (cx, cy-arm-gap), (cx, cy-gap), col, 2, cv2.LINE_AA)
    cv2.line(img, (cx, cy+gap),     (cx, cy+arm+gap), col, 2, cv2.LINE_AA)

    # círculo principal
    r = 18 if locked else 14
    cv2.circle(img, (cx,cy), r, col, 1, cv2.LINE_AA)
    cv2.circle(img, (cx,cy),  2, col, -1)

    # marcas de alcance nos eixos
    for d in [100, 145, 190]:
        for s in [-1,1]:
            cv2.line(img,(cx+s*d,cy-5),(cx+s*d,cy+5),G_MID,1)
            cv2.line(img,(cx-5,cy+s*d),(cx+5,cy+s*d),G_MID,1)

    if locked:
        # anel externo pulsante (círculo maior, meio transparente)
        cv2.circle(img, (cx,cy), r+10, G_DIM, 1, cv2.LINE_AA)
        # "TRACKING" perto da mira
        put_text(img, "TRACKING", (cx+r+14, cy-5), G_BRIGHT, 0.8)


# ── CANTOS DA TELA ───────────────────────────────────────────
def draw_screen_corners(img, w, h):
    for (x,y),(dx,dy) in [((8,8),(1,1)),((w-9,8),(-1,1)),
                           ((8,h-9),(1,-1)),((w-9,h-9),(-1,-1))]:
        cv2.line(img,(x,y),(x+dx*40,y),G_BRIGHT,2,cv2.LINE_AA)
        cv2.line(img,(x,y),(x,y+dy*40),G_BRIGHT,2,cv2.LINE_AA)


# ── SEEKER VIEW ──────────────────────────────────────────────
def draw_seeker(frame, orig_frame, x1, y1, x2, y2):
    fh,fw = frame.shape[:2]
    bw,bh = x2-x1, y2-y1
    px,py = int(bw*ZOOM_PAD), int(bh*ZOOM_PAD)
    rx1=max(0,x1-px); ry1=max(0,y1-py)
    rx2=min(fw,x2+px); ry2=min(fh,y2+py)
    crop = orig_frame[ry1:ry2,rx1:rx2]
    if crop.size == 0: return

    zoomed = cv2.resize(crop,(SEEKER_W,SEEKER_H),interpolation=cv2.INTER_LINEAR)
    k = np.array([[0,-0.4,0],[-0.4,2.6,-0.4],[0,-0.4,0]])
    zoomed = cv2.filter2D(zoomed,-1,k)

    margin = 10
    ox = fw - SEEKER_W - margin
    oy = fh - SEEKER_H - margin

    overlay = frame.copy()
    cv2.rectangle(overlay,(ox-4,oy-20),(ox+SEEKER_W+4,oy+SEEKER_H+4),BLACK,-1)
    cv2.addWeighted(overlay,0.55,frame,0.45,0,frame)
    frame[oy:oy+SEEKER_H, ox:ox+SEEKER_W] = zoomed
    cv2.rectangle(frame,(ox-1,oy-1),(ox+SEEKER_W+1,oy+SEEKER_H+1),G_BRIGHT,1,cv2.LINE_AA)
    put_text(frame,"SEEKER VIEW",(ox,oy-6),G_BRIGHT,FONT_MD)

    sc = (ox+SEEKER_W//2, oy+SEEKER_H//2)
    cv2.line(frame,(sc[0]-14,sc[1]),(sc[0]+14,sc[1]),G_BRIGHT,1,cv2.LINE_AA)
    cv2.line(frame,(sc[0],sc[1]-14),(sc[0],sc[1]+14),G_BRIGHT,1,cv2.LINE_AA)
    cv2.circle(frame,sc,20,G_MID,1,cv2.LINE_AA)
    dashed_rect(frame,sc[0]-20,sc[1]-20,sc[0]+20,sc[1]+20,G_BRIGHT,gap=6)


# ── TARGET BOX ───────────────────────────────────────────────
def draw_target_box(img, x1, y1, x2, y2, tid, conf, primary=False):
    arm = max(14,min(22,(x2-x1)//3))
    dashed_rect(img,x1,y1,x2,y2,G_DIM,gap=7)
    corner_rect(img,x1,y1,x2,y2,G_BRIGHT,arm=arm,thick=2)
    tcx,tcy = (x1+x2)//2,(y1+y2)//2
    cv2.drawMarker(img,(tcx,tcy),G_MID,cv2.MARKER_CROSS,8,1,cv2.LINE_AA)

    label = f"UAV {tid:02d}  {conf:.0%}"
    (lw,lh),_ = cv2.getTextSize(label,FONT,FONT_SM,1)
    cv2.rectangle(img,(x1-1,y1-lh-8),(x1+lw+4,y1),BLACK,-1)
    put_text(img,label,(x1+2,y1-4),G_BRIGHT,FONT_SM)
    if primary:
        put_text(img,"LOCK",(x2-28,y1-4),G_BRIGHT,FONT_SM)


# ── PAINÉIS ───────────────────────────────────────────────────
def draw_panels(img, w, h, targets_info, fps, now):
    left = [
        ("TARGET SELECTOR", G_BRIGHT),
        (f"Targets  : {len(targets_info):02d}", G_MID),
        ("", G_DIM),
        ("TARGET TRACKING", G_BRIGHT),
    ]
    if targets_info:
        t = targets_info[0]
        left += [
            (f"ID       : {t['id']:02d}", G_MID),
            (f"Conf     : {t['conf']:.0%}", G_MID),
            (f"Size     : {t['w']}x{t['h']}px", G_MID),
            (f"Pos XY   : {t['cx']},{t['cy']}", G_MID),
        ]
    else:
        left += [("No target", G_DIM)]
    for i,(text,col) in enumerate(left):
        put_text(img,text,(12,20+i*16),col,FONT_SM)

    right = [
        ("TEST SETTINGS", G_BRIGHT),
        ("Source    : VIDEO",        G_MID),
        ("Seeker    : EO",           G_MID),
        ("Tracker   : ByteTrack",    G_MID),
        ("Model     : YOLOv11x",     G_MID),
        ("", G_DIM),
        ("TRACKER DATA", G_BRIGHT),
        (f"FPS       : {fps:.1f}",   G_MID),
        (f"Time      : {now}",       G_DIM),
    ]
    for i,(text,col) in enumerate(right):
        (lw,_),_ = cv2.getTextSize(text,FONT,FONT_SM,1)
        put_text(img,text,(w-lw-12,20+i*16),col,FONT_SM)

    mid = w//2
    sensor = "SENSOR MODE: EO"
    (sw,_),_ = cv2.getTextSize(sensor,FONT,FONT_SM,1)
    put_text(img,sensor,(mid-sw//2,h-26),G_MID,FONT_SM)
    cmds = "Target Box: Q  |  Lock: C  |  Cancel: V  |  Mode: H  |  Adjust: N"
    (cw,_),_ = cv2.getTextSize(cmds,FONT,0.75,1)
    put_text(img,cmds,(mid-cw//2,h-8),G_DIM,0.75)


# ── HUD PRINCIPAL ────────────────────────────────────────────
def draw_hud(frame, results, fps, orig_frame):
    global _aim_x, _aim_y

    out = frame.copy()          # sem filtro de cor
    h, w = out.shape[:2]

    # inicializa mira no centro na primeira frame
    if _aim_x < 0:
        _aim_x, _aim_y = float(w//2), float(h//2)

    boxes = results[0].boxes
    targets_info = []
    primary_box  = None
    target_cx, target_cy = None, None

    if boxes is not None and boxes.id is not None:
        order = sorted(range(len(boxes)),
                       key=lambda i: float(boxes.conf[i]), reverse=True)
        for rank, i in enumerate(order):
            x1,y1,x2,y2 = map(int, boxes.xyxy[i])
            tid  = int(boxes.id[i])
            conf = float(boxes.conf[i])
            tcx,tcy = (x1+x2)//2, (y1+y2)//2

            trails[tid].append((tcx,tcy))
            if len(trails[tid]) > TRAIL_LEN:
                trails[tid].pop(0)
            if len(trails[tid]) > 1:
                draw_trail(out, trails[tid])

            primary = (rank == 0)
            draw_target_box(out, x1,y1,x2,y2, tid, conf, primary)

            if primary:
                primary_box  = (x1,y1,x2,y2)
                target_cx    = tcx
                target_cy    = tcy

            targets_info.append({
                "id":tid,"conf":conf,"cx":tcx,"cy":tcy,"w":x2-x1,"h":y2-y1
            })

    # ── lerp: mira se move suavemente em direção ao alvo ──────
    locked = target_cx is not None
    if locked:
        # suavização exponencial — quanto maior AIM_SMOOTH mais lento
        _aim_x = _aim_x * AIM_SMOOTH + target_cx * (1.0 - AIM_SMOOTH)
        _aim_y = _aim_y * AIM_SMOOTH + target_cy * (1.0 - AIM_SMOOTH)
    else:
        # sem alvo: retorna lentamente ao centro
        _aim_x = _aim_x * AIM_SMOOTH + (w//2) * (1.0 - AIM_SMOOTH)
        _aim_y = _aim_y * AIM_SMOOTH + (h//2) * (1.0 - AIM_SMOOTH)

    aim_cx = int(round(_aim_x))
    aim_cy = int(round(_aim_y))

    draw_screen_corners(out, w, h)
    draw_crosshair(out, aim_cx, aim_cy, w, h, locked=locked)

    # seeker usa frame original (sem HUD)
    if primary_box is not None:
        draw_seeker(out, orig_frame, *primary_box)

    now = datetime.now(timezone.utc).strftime("%H:%M:%S UTC")
    draw_panels(out, w, h, targets_info, fps, now)

    return out


# ── ABRIR VÍDEO ─────────────────────────────────────────────
cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError("Erro ao abrir vídeo")

W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*'mp4v'),
    FPS, (W,H)
)

print("Processando vídeo...")
frame_n = 0
fps_buf = []

while True:
    ret, frame = cap.read()
    if not ret: break

    frame_n += 1
    orig = frame.copy()

    t0 = time.time()
    results = model.track(
        frame, conf=CONF, persist=True,
        tracker="bytetrack.yaml", verbose=False
    )
    fps_buf.append(time.time()-t0)
    if len(fps_buf) > 30: fps_buf.pop(0)
    fps_now = 1.0 / (sum(fps_buf)/len(fps_buf))

    out_frame = draw_hud(frame, results, fps_now, orig)
    writer.write(out_frame)

    if frame_n % 30 == 0:
        print(f"  Frame {frame_n}  |  FPS médio: {fps_now:.1f}")

cap.release()
writer.release()
print(f"\n✓ Vídeo salvo em: {OUTPUT_VIDEO}")

Carregando modelo YOLO...
✓ Modelo carregado
Processando vídeo...
  Frame 30  |  FPS médio: 15.7
  Frame 60  |  FPS médio: 27.4
  Frame 90  |  FPS médio: 27.3
  Frame 120  |  FPS médio: 27.2
  Frame 150  |  FPS médio: 25.9
  Frame 180  |  FPS médio: 26.9
  Frame 210  |  FPS médio: 27.1
  Frame 240  |  FPS médio: 27.3
  Frame 270  |  FPS médio: 27.2

✓ Vídeo salvo em: /content/uav_tracking_result.mp4
